# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their field `@id`s.

In [ ]:
# List all record sets by their @id
print("Available record sets (by @id):")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")

# For each record set, list its fields by @id
print("\nRecord set fields by @id:")
record_set_fields = dict()
for rs in record_sets:
    print(f"\nRecord Set '{rs.name}' (@id={rs.id}):")
    field_ids = []
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
        field_ids.append(field.id)
    record_set_fields[rs.id] = field_ids


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract all record sets
import warnings
warnings.filterwarnings('ignore')

dataframes = dict()
for rs in record_sets:
    try:
        print(f"Loading records for record set: {rs.id} ...")
        df = pd.DataFrame(list(dataset.records(record_set=rs.id)))
        dataframes[rs.id] = df
        print(f" - Columns: {df.columns.tolist()}")
        print(f" - Number of records: {len(df)}\n")
    except Exception as e:
        print(f"Could not load record set {rs.id} due to error: {e}")

# For demonstration, let's pick the first available record set
if record_sets:
    example_record_set_id = record_sets[0].id
    print(f"Columns in '{example_record_set_id}':", dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This includes removing outliers, transforming data distributions, or grouping data by key attributes. You should adapt this for the actual numeric and grouping fields in the dataset.

In [ ]:
# Example: Use the first record set and pick a numeric field
record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Find a likely numeric field by data type if possible
numeric_field_id = None
group_field_id = None
for field in metadata.record_set(record_set_id).fields:
    # Check data type, prefer Float, Integer, Number
    dt = getattr(field, 'data_type', None)
    if not numeric_field_id and dt in ('Float', 'Integer', 'Number'):
        numeric_field_id = field.id
    # Also pick a string/categorical field for grouping
    if not group_field_id and dt == 'Text':
        group_field_id = field.id

if numeric_field_id is not None:
    print(f"Selected numeric field for EDA: {numeric_field_id}")
else:
    print("No numeric field detected in the record set.")

threshold = None

# If a numeric field is found, proceed
if numeric_field_id in df.columns:
    # Try converting to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Use median as a threshold (for demo)
    threshold = df[numeric_field_id].median()
    
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Column {numeric_field_id} not available for numeric EDA.")

# Grouping by a categorical field
if group_field_id is not None and group_field_id in df.columns:
    # Group by the text field and compute mean of available numeric
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable group-by field detected or present in columns.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple histogram and boxplot for the numeric field (if available)
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        top_groups = df[group_field_id].value_counts().index[:6]
        plt.figure(figsize=(12,6))
        sns.boxplot(data=df[df[group_field_id].isin(top_groups)], x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric column available for plotting.")


## 6. Conclusion
We have used the `mlcroissant` library to dynamically explore the structure and contents of a FAIR Croissant-formatted dataset. This included:
- Querying metadata and understanding its structure by `@id`
- Loading record sets and fields by their unique identifiers
- Extracting dataframes for flexible analysis
- Performing basic EDA, such as filtering and normalizing a sample numeric field, and simple grouping
- Visualizing data distributions where feasible

This workflow provides a reproducible foundation for deeper scientific analysis. For custom insights, continue exploring the full range of record sets, fields, and Croissant metadata using the presented code patterns.